<a href="https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliza1800/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

The key fields are strongly skewed rather than evenly distributed. Impressions have a long right tail, with a median of 731 compared with a maximum of 517,715. CTR and engagement rate also contain extreme values, while trend_pct has a very wide range and a large positive outlier, so raw values should be interpreted cautiously.

In [10]:
!git clone https://github.com/aliza1800/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [11]:
import pandas as pd

df = pd.read_csv(
    "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

print("Dataset loaded:", df.shape)
print(df.columns.tolist())

Dataset loaded: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [12]:

key_fields = [
    "impressions_90d",
    "ctr",
    "engagement_rate",
    "trend_pct"
]

for col in key_fields:
    print(f"\n{'='*50}")
    print(col)
    print(df[col].describe())


impressions_90d
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64

ctr
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64

engagement_rate
count    30000.000000
mean         2.534520
std          8.310096
min          0.000000
25%          0.000000
50%          0.000000
75%          1.350000
max        100.000000
Name: engagement_rate, dtype: float64

trend_pct
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Signal #1 — HIGH_IMPRESSIONS: MIXED. The observed median engagement rate is 0.0 for both high- and lower-impression groups, so high impressions alone do not clearly indicate weaker engagement.

Signal #2 — LOW_CTR: CONFIRMED. The low-CTR group had a much lower median impression level (129) than the higher-CTR group (2493.5), supporting a measurable relationship in this test.

Signal #3 — DECLINING: MIXED. Declining pages had a higher median impression level (1159) than non-declining pages (724), so declining performance does not consistently indicate lower visibility in this test.

In [13]:
# Signal tests: compare outcomes across simple signal groups

# Signal 1: High impressions vs lower impressions
impression_cutoff = df["impressions_90d"].median()
high_impressions = df[df["impressions_90d"] >= impression_cutoff]
low_impressions = df[df["impressions_90d"] < impression_cutoff]

print("SIGNAL 1: HIGH IMPRESSIONS")
print("High-impression median engagement:", high_impressions["engagement_rate"].median())
print("Low-impression median engagement:", low_impressions["engagement_rate"].median())

# Signal 2: Low CTR vs higher CTR
ctr_cutoff = df["ctr"].median()
low_ctr = df[df["ctr"] <= ctr_cutoff]
high_ctr = df[df["ctr"] > ctr_cutoff]

print("\nSIGNAL 2: LOW CTR")
print("Low-CTR median impressions:", low_ctr["impressions_90d"].median())
print("High-CTR median impressions:", high_ctr["impressions_90d"].median())

# Signal 3: Declining trend vs non-declining
declining = df[df["trend_pct"] < 0]
non_declining = df[df["trend_pct"] >= 0]

print("\nSIGNAL 3: DECLINING")
print("Declining pages:", len(declining))
print("Non-declining pages:", len(non_declining))
print("Declining median impressions:", declining["impressions_90d"].median())
print("Non-declining median impressions:", non_declining["impressions_90d"].median())

SIGNAL 1: HIGH IMPRESSIONS
High-impression median engagement: 0.0
Low-impression median engagement: 0.0

SIGNAL 2: LOW CTR
Low-CTR median impressions: 129.0
High-CTR median impressions: 2493.5

SIGNAL 3: DECLINING
Declining pages: 19715
Non-declining pages: 6897
Declining median impressions: 1159.0
Non-declining median impressions: 724.0


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Verdict: CONFIRMED for visibility, with a limitation. The low-CTR group had a much lower median impressions level (74) than the other pages (2802.5), supporting the assumption that zero-CTR pages have weaker observed search visibility. However, median engagement rate was 0.0 in both groups, so this test does not support a low-CTR-to-low-engagement relationship.

In [14]:
# Flag-linked test: LOW_CTR signal

# Define low CTR as the bottom 25% of observed CTR values
ctr_cutoff = df["ctr"].quantile(0.25)

low_ctr_group = df[df["ctr"] <= ctr_cutoff]
other_ctr_group = df[df["ctr"] > ctr_cutoff]

print("LOW_CTR flag-linked test")
print("-" * 40)
print("CTR cutoff:", ctr_cutoff)
print("Low-CTR pages:", len(low_ctr_group))
print("Other pages:", len(other_ctr_group))

print("\nMedian impressions:")
print("Low-CTR:", low_ctr_group["impressions_90d"].median())
print("Other:", other_ctr_group["impressions_90d"].median())

print("\nMedian engagement rate:")
print("Low-CTR:", low_ctr_group["engagement_rate"].median())
print("Other:", other_ctr_group["engagement_rate"].median())

LOW_CTR flag-linked test
----------------------------------------
CTR cutoff: 0.0
Low-CTR pages: 13212
Other pages: 16788

Median impressions:
Low-CTR: 74.0
Other: 2802.5

Median engagement rate:
Low-CTR: 0.0
Other: 0.0


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Content teams should treat low CTR as a useful directional signal for identifying pages with weak observed visibility, but should not use it alone to decide on a refresh. The audit shows that engagement and declining-performance signals need additional context because their relationships with visibility were mixed in these tests.

In [15]:
print("Practical signal summary")
print("-" * 40)
print("LOW_CTR: supported for lower observed impressions.")
print("HIGH_IMPRESSIONS: mixed because median engagement was 0.0 in both groups.")
print("DECLINING: mixed because declining pages had higher median impressions.")


Practical signal summary
----------------------------------------
LOW_CTR: supported for lower observed impressions.
HIGH_IMPRESSIONS: mixed because median engagement was 0.0 in both groups.
DECLINING: mixed because declining pages had higher median impressions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.